# Figure S4b–c — program CRT calibration and biological-replicate effect concordance

This notebook is restricted to **Figure S4b and Figure S4c**. It does not reconstruct either panel with generic plotting functions. Instead, it runs the previous notebook that produced the CRT and replicate-effect analyses:

```text
/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/
iPSC_EC_QQ_full_covariates_with_replicate_correlation_bundle/
iPSC_EC_QQ_full_covariates_with_replicate_correlation.ipynb
```

Frozen executed-source SHA-256:

```text
6e80625abd9b5e71d45da6dd19e888898318d39a159a7ac35600fc1d85489e87
```

The repository also contains the recovered code-source copy under `recovery/raw_downloads/iPSC_EC_QQ_full_covariates_with_replicate_correlation_bundle/`, frozen as Git blob `04e5c97707b847e09a89f7ca7b4ca2c83bd3b015`.

The source notebook's programDE/CRT workflow, covariates, NTC pseudo-target construction, beta-effect calculation, filters, plotnine styling, facet labels, colors, axes, and PDF/SVG/EPS/PNG exports are retained without scientific or visual changes.

- **S4b:** empirical CRT Q–Q plots for real targets and NTC pseudo-targets, separately for D0–D3.
- **S4c:** biological-replicate scatterplots of matched real `target_name × program_name` beta effects, with the source notebook's Pearson and Spearman summaries.


In [ ]:
from pathlib import Path
import json
import os
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing src/cc_perturbseq. "
        "Run this notebook from within the cc-perturbseq-manuscript-analysis checkout."
    )


REPO_ROOT = _find_repo_root()
for support_path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(support_path) not in sys.path:
        sys.path.insert(0, str(support_path))

from src.cc_perturbseq.figure_notebook import (
    initialize_figure,
    load_figure_manifest,
    panel_directory,
    write_receipt,
)
from src.cc_perturbseq.s4_source_runner import (
    ARCHIVE_SOURCE_GIT_BLOB_SHA1,
    OAK_SOURCE_NOTEBOOK,
    OAK_SOURCE_SHA256,
    REUSE_ENVIRONMENT_VARIABLE,
    SOURCE_ENVIRONMENT_VARIABLE,
    collect_s4_source_outputs,
    resolve_s4_source_notebook,
)
from src.cc_perturbseq.source_notebook_runner import execute_notebook

FIGURE = load_figure_manifest("Figure_S04")
ctx = initialize_figure(FIGURE)
PANEL_B_DIR = panel_directory(ctx, "b")
PANEL_C_DIR = panel_directory(ctx, "c")
FIGURE_DIR = PANEL_B_DIR.parent

print("Repository:", REPO_ROOT)
print("Figure output:", FIGURE_DIR)
print("Frozen Oak source:", OAK_SOURCE_NOTEBOOK)
print("Frozen Oak SHA-256:", OAK_SOURCE_SHA256)
print("Frozen archive Git blob:", ARCHIVE_SOURCE_GIT_BLOB_SHA1)


## Resolve the frozen previous notebook

The preferred source is the executed Sherlock notebook at its original `programDE/notebooks/...` location. The repository's recovered code-source copy is used only when the exact Oak source is unavailable. A different notebook is rejected rather than silently substituted.


In [ ]:
explicit_source = os.environ.get(SOURCE_ENVIRONMENT_VARIABLE)
source = resolve_s4_source_notebook(REPO_ROOT, explicit_source)
source_path = Path(source.path)

source_record = {
    "source_notebook": source.path,
    "source_kind": source.source_kind,
    "verification_algorithm": source.verification_algorithm,
    "expected_hash": source.expected_hash,
    "observed_hash": source.observed_hash,
    "source_logic_changed": False,
    "source_style_changed": False,
    "panels": ["b", "c"],
}
print(json.dumps(source_record, indent=2))


## Execute the previous notebook

By default this runs the exact source notebook from top to bottom in its own working directory. This preserves the original relative paths and output layout. To collect already-existing outputs without rerunning the CRT, set:

```bash
export CCPERTURBSEQ_FIGURE_S04_REUSE_SOURCE_OUTPUTS=1
```

No alternate plotting implementation is used in either mode.


In [ ]:
reuse_existing = os.environ.get(REUSE_ENVIRONMENT_VARIABLE, "0") == "1"
executed_notebook = FIGURE_DIR / "Figure_S04bc_exact_source_executed.ipynb"

if reuse_existing:
    execution_status = "EXACT_SOURCE_OUTPUTS_REUSED"
    print(
        f"{REUSE_ENVIRONMENT_VARIABLE}=1; not rerunning the source notebook. "
        "Existing source outputs will be validated and copied."
    )
else:
    execute_notebook(
        source_path,
        executed_notebook,
        working_directory=source_path.parent,
        timeout=172_800,
    )
    execution_status = "EXACT_SOURCE_NOTEBOOK_EXECUTED"
    print("Executed source notebook:", executed_notebook


## Collect S4b and S4c outputs

The files copied below are those written by the previous notebook itself. The driver verifies that vector plots and the corresponding source-data tables exist before writing figure receipts.


In [ ]:
collected = collect_s4_source_outputs(
    source,
    panel_b_directory=PANEL_B_DIR,
    panel_c_directory=PANEL_C_DIR,
)

provenance = {
    **source_record,
    "execution_status": execution_status,
    "executed_notebook": str(executed_notebook) if executed_notebook.is_file() else None,
    "source_model_output_root": collected["source"]["model_output_root"],
    "panel_b_outputs": collected["panel_b"],
    "panel_c_outputs": collected["panel_c"],
}
provenance_path = FIGURE_DIR / "Figure_S04bc_exact_source_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

common_inputs = {
    "source_notebook": source.path,
    "source_hash": source.observed_hash,
    "source_provenance": str(provenance_path),
}

write_receipt(
    ctx,
    "b",
    status=execution_status,
    inputs=common_inputs,
    outputs={
        **collected["panel_b"],
        "executed_source_notebook": str(executed_notebook)
        if executed_notebook.is_file()
        else "reused_existing_source_outputs",
        "provenance": str(provenance_path),
    },
    parameters={
        "source_logic_changed": False,
        "source_style_changed": False,
        "analysis": "day-specific empirical CRT Q-Q",
        "series": ["Real targets", "NTC pseudo-targets"],
        "days": ["d0", "d1", "d2", "d3"],
        "B": 10000,
        "covariates": [
            "pct_counts_mt",
            "biological_sample",
            "log1p_total_counts",
            "log1p_guides_per_cell",
            "log1p_guide_umi_counts",
            "log1p_n_genes_by_counts",
        ],
    },
)

write_receipt(
    ctx,
    "c",
    status=execution_status,
    inputs=common_inputs,
    outputs={
        **collected["panel_c"],
        "executed_source_notebook": str(executed_notebook)
        if executed_notebook.is_file()
        else "reused_existing_source_outputs",
        "provenance": str(provenance_path),
    },
    parameters={
        "source_logic_changed": False,
        "source_style_changed": False,
        "effect": "programDE betas_df",
        "point_unit": "real target_name × program_name",
        "matching": "inner join across biological samples within day",
        "excluded_targets": ["SAFE", "non-targeting", "NTC"],
        "correlations": ["Pearson", "Spearman"],
        "B_REP": 127,
        "minimum_cells_per_replicate": 200,
    },
)

print("Figure S4b outputs:")
for path in collected["panel_b"].values():
    print(" ", path)
print("Figure S4c outputs:")
for path in collected["panel_c"].values():
    print(" ", path)
